In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error, mean_absolute_error

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import HuberRegressor, ElasticNet
import xgboost as xgb
import catboost as cb

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [3]:
# read data

train_data_raw = pd.read_csv('.data/train.csv')
test_data_raw = pd.read_csv('.data/test.csv')

In [4]:
# EDA: train data first look

# print(train_data_raw.head(10))
train_data_raw.info()
# train_data_raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [5]:
print(train_data_raw.head(10))

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   
5   6          50       RL         85.0    14115   Pave   NaN      IR1   
6   7          20       RL         75.0    10084   Pave   NaN      Reg   
7   8          60       RL          NaN    10382   Pave   NaN      IR1   
8   9          50       RM         51.0     6120   Pave   NaN      Reg   
9  10         190       RL         50.0     7420   Pave   NaN      Reg   

  LandContour Utilities LotConfig LandSlope Neighborhood Condition1  \
0         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
1         Lvl    AllPub       FR2       Gtl

In [6]:
# EDA: test data first look

# test_data_raw.head(10)
test_data_raw.info()
# test_data_raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   str    
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   str    
 6   Alley          107 non-null    str    
 7   LotShape       1459 non-null   str    
 8   LandContour    1459 non-null   str    
 9   Utilities      1457 non-null   str    
 10  LotConfig      1459 non-null   str    
 11  LandSlope      1459 non-null   str    
 12  Neighborhood   1459 non-null   str    
 13  Condition1     1459 non-null   str    
 14  Condition2     1459 non-null   str    
 15  BldgType       1459 non-null   str    
 16  HouseStyle     1459 non-null   str    
 17  OverallQual    1459 non-null   int64  
 18  OverallCond    1459

In [ ]:
# data cleaning
###################


# drop id column
train_data = train_data_raw.drop(columns=['Id'])
test_data = test_data_raw.drop(columns=['Id'])

# there is couple columns that have the numerical dtype but actually categorical, so we need to convert them to str or object dtype

columns_to_convert_n2s = ["MSSubClass"]

for col in columns_to_convert_n2s:
    train_data[col] = train_data[col].astype(str)
    test_data[col] = test_data[col].astype(str)

# seperate numerical and categorical columns, and target column
target_col = "SalePrice"

# all columns except target column are features
feature_cols = [c for c in train_data.columns if c != target_col]

numerical_cols = train_data[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = train_data[feature_cols].select_dtypes(include=["object", "string"]).columns.tolist()

# handle nan, none missing values, etc.
# for almost all columns, nan means that feature is not present, so it can be filled with 0 for numerical columns, and "None" for categorical columns

for col in numerical_cols:
    train_data[col] = train_data[col].fillna(0)
    test_data[col] = test_data[col].fillna(0)

for col in categorical_cols:
    train_data[col] = train_data[col].fillna("None")
    test_data[col] = test_data[col].fillna("None")

np.int64(0)

In [11]:
# double check if there is any missing value left

print(train_data.isnull().sum().sum())
print(test_data.isnull().sum().sum())

0
0


# some notes on data

- year built and year remod needs special scaling
- for some columns NA means lack of the said feature
- datetime conversion from MoSold: Month Sold (MM), YrSold: Year Sold (YYYY)

In [ ]:
# data preprocessing
#####################



In [ ]:
kernel = ConstantKernel() * RBF() + WhiteKernel()

GaussianProcessRegressor(kernel=kernel, normalize_y=True)